# AIR pilot 3: Rao-Blackwellised transform latent θ_c vs a clean A-only PFN

## Description

<!-- Written 2026-11-03, before the first pilot-3 run. Plan: docs/a2b_air_pilot1_review_2026-11-03.md (the user's review of
pilot 1), amended below with the user's decisions. Pilot 1-2: notebook 10. Issue #19. -->

**Question.** Does AIR beat a clean A-only PFN once its two pilot-1 weaknesses are fixed (an underfit proposal; sharp per-draw
predictives that make small K lossy), and does a noisy view of the transform (Σ > 0) help at fixed K?

**Idea (review §3).** Augment the model with θ_c = θ̃ + √Σ ε, a latent the data never see. Then exactly
p(y* | A, B) = ∫ p(θ_c | A, B) p(y* | A, B, θ_c) dθ_c, with prior p(θ_c) = N(0, (1 + Σ) I).
- **Decoder:** sees B moved by T_{θ_c}⁻¹ and is told Σ; it is trained on data generated from the true θ̃.
- **Proposal:** trained by NPE on θ_c.
- **Weights:** N(θ_c; 0, (1 + Σ) I) p̂(A | B, θ_c) / q(θ_c).

Σ = 0 is pilot-1 AIR. Larger Σ makes each draw's predictive smoother (fewer holes at small K) and the posterior over θ_c wider (easier to fit with q),
at the cost of transform information per draw.

**Setup.**
- **Prior:** as pilot 2, d = 1, H0 (h = id), Z0 (always related), g normalised by the global prior ECDF (`task.g_norm=ecdf`, the repo standard).
  The pilot-2 decoder already shows transfer under this prior: validation NLL at n_A = 5 is −2.516 (teacher) vs −1.835 (A-only), sweep
  `pilot_20261103-090459`. Pilot 2's evaluation is not used to choose anything here.
- **Σ ∈ {0, 0.1, 0.3, 1}** (covariance Σ·I in θ̃ units).
- **Decoder:** one Σ-conditioned network (`Decoder(sig_cond=True)`), with Σ embedded as (Σ, log(Σ + 0.01)) and added to every token.
  - Trained with Σ drawn uniformly per task from that list, and J ∈ {2, 4, 8, 16}.
  - The null head is the same network with B masked.
- **Proposal:** one Σ-conditioned q(θ_c | A, B, Σ) per J ∈ {2, 16}, encoder E0 + NSF as in pilot 1, gradient clip 20 (was 1.0).
- **Baseline:** a **clean A-only PFN**, the same architecture trained separately on A only, with the same plateau rule.
- **Training:** AdamW, lr 3·10⁻⁴, 500 warm-up steps. Then a constant rate until the validation NLL improves by < 0.01 over 5000 steps (checked
  from step 10 000), then a 5000-step cosine anneal and stop. 100 000 steps at most. Decoder and A-only clip: 1.0.
- **Seeds:** 0, 1, 2 for every trained network. All seeds are scored on the same 256 evaluation tasks per (J, n_A).
- **Evaluation:** J ∈ {2, 16}, n_A ∈ {1, 2, 3, 5, 10}, 256 tasks, 32 queries each.
  - q-IS: K ∈ {8, 32, 128, 512, 2048} at Σ = 0; K ∈ {8, 32, 128} at Σ > 0. QMC (scrambled Sobol, deterministic mixture split) at K ∈ {8, 32}.
  - **IS-ref:** at Σ = 0, K = 4096. J = 2 uses prior IS; J = 16 uses q-IS, because prior IS collapses there. The exact predictive does not depend on Σ,
    so one reference serves every Σ.
  - Teacher (true θ̃, Σ = 0); G-ref (dense-B grid, J = 2) reported as a floor only.
  - Calibration of q: SBC ranks of the true θ_c among 255 draws, and HPD coverage, per (J, Σ, n_A).
  - At Σ = 0, J = 2, for 128 tasks on a 61² grid: the decoder-implied posterior π̂_D ∝ p(θ̃) p̂(A | B, θ̃), with its marginal PIT of the true
    θ̃, and KL(π̂_D ‖ q).

**Pre-registered criteria** (nats per query; means over 3 seeds unless stated; J = 2 unless stated).

| ID | Check | Pass |
|---|---|---|
| P3.1 | q at Σ = 0 (trained to plateau, clip 20) | SBC KS p > 0.01 per coordinate; 50 / 90 % HPD coverage within ±5 points; median KL(π̂_D ‖ q) < 0.1; at every n_A |
| P3.2 | Decoder-implied posterior calibration, Σ = 0 | PIT KS p > 0.01 per coordinate, at every n_A |
| P3.3 | K-curve, Σ = 0 | q-IS(128) − IS-ref ≤ 0.03 at n_A ≥ 2 and ≤ 0.07 at n_A = 1 |
| P3.4 | **Primary.** AIR at its best Σ vs the clean A-only PFN, K ∈ {8, 32} | for each K: the best Σ (lowest mean NLL over n_A ≥ 2) beats the clean A-only PFN by ≥ 0.05 at every n_A ≥ 2, with the seed-mean difference > 2 standard errors across seeds |
| P3.5 | J scaling at the best Σ, K = 128 | (q-IS − IS-ref) at J = 16 minus the same at J = 2 is ≤ 0.05, at the n_A where the J = 16 IS-ref has median ESS ≥ 500 |
| P3.6 | QMC vs i.i.d. at equal K | reported, no threshold |

Reported with P3.4, no threshold: whether the best Σ is > 0, and its paired seed difference from Σ = 0 at each K.

**Decision rule** (exclusive):
1. **P3.4 passes and the best Σ is > 0 at K = 32** → continue with AIR at that Σ to h (V7) and d = 2.
2. **P3.4 passes and the best Σ is 0 at K = 32** → continue with AIR at Σ = 0; the augmentation is not needed.
3. **P3.4 fails** → the explicit latent does not pay in d = 1. Run the J = 16 and d = 2 cells before abandoning it.

**Deviations from the review** (decided 2026-11-03):
- Σ = ∞ is dropped (the user). For large Σ, B still enters warped by an extreme T_{θ_c}⁻¹, so it is not a plain joint PFN.
- The baseline is the clean A-only PFN, not a direct joint [A, B] PFN (the user).
- One Σ-conditioned decoder and one Σ-conditioned proposal per J (Σ as input), not one network per Σ.
- The K-curve is run at Σ = 0 only; Σ > 0 at K ≤ 128.
- P3.5 uses K = 128 and a q-IS reference at J = 16 (the review's K = 32 conflicted with its own §2.5, and prior IS collapses at J = 16).
- §2.3's per-point permutation criterion is not run: the functional check (ESS unchanged with 8 orders) already passed in pilot 1.
- The 3 seeds vary training only; evaluation tasks are shared, so seed SEs measure training variability.

**Limits.**
- d = 1, H0, Z0 only.
- G-ref knows g exactly and is a floor, not a target.
- The best Σ is selected on the same tasks it is scored on (4 candidates; reported, not hidden).
- NLLs are not comparable with pilot 1 (different prior) or runs 1–2 (different bins).


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/air_rb/<sweep_id>"  # set after `just snapshot runs/air_rb/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.